# DLAttack on Amazon Reviews 2023 (Video Games, 5-core)

**Reference:** Huang et al., *Data Poisoning Attacks to Deep Learning Based Recommender Systems* (NDSS 2021) — [arXiv:2101.02644](https://arxiv.org/abs/2101.02644)

Large-scale replication using a **TorchRec Two-Tower** recommender on Amazon Reviews 2023 Video Games (~95K users, ~26K items, ~815K interactions) — **16x more users and 7x more items** than MovieLens-1M.

| Metric | Video_Games 5-core | MovieLens-1M |
|--------|-------------------|--------------|
| Users | ~94,800 | ~6,000 |
| Items | ~25,600 | ~3,700 |
| Interactions | ~814,600 | ~1M |
| Density | ~0.034% | ~4.5% |

**Pipeline:**
1. **Baseline training** — Two-Tower with Adam optimizer
2. **DLAttack** — surrogate optimization + fake user injection (50 fake users = ~0.05% poison ratio)
3. **Evaluation** — target item promotion + overall metrics
4. **Detection** — TIA with threshold sweep + AUC-ROC

**Requirements:** GPU runtime (Runtime > Change runtime type > A100 GPU)

---

## Phase 0: Setup

In [ ]:
# Clone the repo (skip if already cloned), pull latest changes
import os
if not os.path.exists("embdguard"):
    !git clone https://github.com/aliafzal/embdguard.git
%cd /content/embdguard/dlattack_research
!git pull

In [ ]:
# Install dependencies
!pip install -q torchrec fbgemm-gpu datasets

In [ ]:
# Verify environment
import torch
import torchrec
import numpy as np
import pandas as pd

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch:    {torch.__version__}")
print(f"torchrec: {torchrec.__version__}")
print(f"device:   {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
os.makedirs("checkpoints/amazon", exist_ok=True)
os.makedirs("results/amazon", exist_ok=True)

## Phase 1: Load Amazon Reviews 2023 (Video Games, 5-core)

In [ ]:
from src.amazon_dataset import load_amazon_reviews
from src.dataset import split_data

df, n_users, n_items, user_map, item_map = load_amazon_reviews(
    category="Video_Games", kcore=5
)
train_df, test_df = split_data(df)

print(f"\nUsers: {n_users:,}  |  Items: {n_items:,}")
print(f"Train: {len(train_df):,} interactions  |  Test: {len(test_df):,} interactions")

In [ ]:
# Quick data summary
interactions_per_user = train_df.groupby("user_id").size()
interactions_per_item = train_df.groupby("item_id").size()

print(f"Interactions per user — mean: {interactions_per_user.mean():.1f}, "
      f"median: {interactions_per_user.median():.0f}, "
      f"min: {interactions_per_user.min()}, max: {interactions_per_user.max()}")
print(f"Interactions per item — mean: {interactions_per_item.mean():.1f}, "
      f"median: {interactions_per_item.median():.0f}, "
      f"min: {interactions_per_item.min()}, max: {interactions_per_item.max()}")

## Phase 2: Model Sanity Check

In [ ]:
from src.model import build_ebc, TwoTower, TwoTowerTrainTask, make_kjt

# Quick forward pass test
test_ebc = build_ebc(n_users=100, n_items=200, embedding_dim=32, device=torch.device("cpu"))
test_model = TwoTower(test_ebc, layer_sizes=[64, 32], device=torch.device("cpu"))

u = torch.tensor([0, 1, 2, 5, 7])
i = torch.tensor([10, 20, 30, 40, 50])
kjt = make_kjt(u, i)
user_emb, item_emb = test_model(kjt)
scores = (user_emb * item_emb).sum(dim=1)

print(f"Scores: {scores.detach().tolist()}")
print(f"User emb shape: {user_emb.shape}, Item emb shape: {item_emb.shape}")

train_task = TwoTowerTrainTask(test_model)
labels = torch.tensor([1.0, 0.0, 1.0, 0.0, 1.0])
loss, _ = train_task(kjt, labels)
print(f"Train task loss: {loss.item():.4f}")

del test_model, test_ebc, train_task
print("\nTorchRec Two-Tower + KJT forward pass OK")

## Phase 3: Train Clean Baseline

Train the Two-Tower model on clean Amazon data. Batch size scaled up to 8192 for the larger dataset.

In [ ]:
# === CONFIGURATION ===
EMBEDDING_DIM = 64
LAYER_SIZES = [128, 64]
BASELINE_EPOCHS = 30
BATCH_SIZE = 8192
LR = 0.001
N_NEG = 4

In [ ]:
from src.model import make_optimizer
from src.train import train
from src.evaluate import evaluate

ebc = build_ebc(n_users, n_items, EMBEDDING_DIM, device=DEVICE)
two_tower = TwoTower(ebc, layer_sizes=LAYER_SIZES, device=DEVICE)
model = TwoTowerTrainTask(two_tower)
optimizer = make_optimizer(model, lr=LR)

eval_fn = lambda m: evaluate(
    m, test_df, train_df, n_items, n_neg=99, k=10, device=str(DEVICE)
)

print(f"=== Baseline Training | {n_users:,} users, {n_items:,} items ===")
print(f"    epochs={BASELINE_EPOCHS}, batch={BATCH_SIZE}, lr={LR}, "
      f"embedding_lr=0.1, neg={N_NEG}\n")

history = train(
    model, optimizer, train_df, n_items,
    epochs=BASELINE_EPOCHS, batch_size=BATCH_SIZE,
    n_neg=N_NEG, device=str(DEVICE),
    eval_fn=eval_fn,
    save_path="checkpoints/amazon/baseline.pt",
)

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

epochs_list = [h[0] for h in history]
losses = [h[1] for h in history]
eval_epochs = [h[0] for h in history if h[2]]
hr_values = [h[2]["HR@K"] for h in history if h[2]]
ndcg_values = [h[2]["NDCG@K"] for h in history if h[2]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_list, losses, "b-o", markersize=3)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train Loss")
ax1.set_title("Training Loss (Amazon Video Games)")
ax1.grid(True, alpha=0.3)

ax2.plot(eval_epochs, hr_values, "g-o", label="HR@10", markersize=5)
ax2.plot(eval_epochs, ndcg_values, "r-s", label="NDCG@10", markersize=5)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("Evaluation Metrics")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/amazon/baseline_training.png", dpi=150, bbox_inches="tight")
plt.show()

final = history[-1][2]
if final:
    print(f"\nFinal Baseline: HR@10={final['HR@K']:.4f}, NDCG@10={final['NDCG@K']:.4f}")

## Phase 4: DLAttack — Targeted Data Poisoning

Select a mid-popularity target item and run 5 rounds of DLAttack, injecting 10 fake users per round (50 total = ~0.05% poison ratio).

Scaled parameters vs MovieLens-1M:
- `m`: 5 → 10 (fake users/round)
- `n_filler`: 30 → 50 (filler items per fake user)
- `n_optim_steps`: 200 → 300 (optimization steps)
- `max_count`: 100 → 200 (target item popularity ceiling)

In [ ]:
# === ATTACK CONFIGURATION ===
ATTACK_ROUNDS = 5
FAKE_USERS_PER_ROUND = 10    # m = 10 -> 50 total fake users
N_FILLER = 50                 # filler items per fake user
N_OPTIM_STEPS = 300           # gradient steps per fake user
RETRAIN_EPOCHS = 10           # epochs when retraining on poisoned data

In [ ]:
# Select target item (mid-popularity: 20-200 interactions)
item_counts = train_df["item_id"].value_counts()
mid_items = item_counts[(item_counts > 20) & (item_counts < 200)].index.tolist()
target_item = int(mid_items[0])

print(f"Target item ID: {target_item}")
print(f"Training interactions: {item_counts[target_item]}")
print(f"Mid-popularity candidates: {len(mid_items)} items")

In [ ]:
import json
from src.attack import run_dlattack

# Build model on real device, load baseline weights
attack_ebc = build_ebc(n_users, n_items, EMBEDDING_DIM, device=DEVICE)
attack_tt = TwoTower(attack_ebc, layer_sizes=LAYER_SIZES, device=DEVICE)
attack_model = TwoTowerTrainTask(attack_tt)

baseline_state = torch.load(
    "checkpoints/amazon/baseline.pt", map_location=DEVICE, weights_only=False
)
attack_model.load_state_dict(baseline_state, strict=False)
attack_optimizer = make_optimizer(attack_model, lr=LR)

eval_fn = lambda m: evaluate(
    m, test_df, train_df, n_items, n_neg=99, k=10, device=str(DEVICE)
)

attack_results, poisoned_train, attack_model, attack_optimizer = run_dlattack(
    model=attack_model,
    optimizer=attack_optimizer,
    train_df=train_df,
    test_df=test_df,
    n_users=n_users,
    n_items=n_items,
    target_item_id=target_item,
    embedding_dim=EMBEDDING_DIM,
    layer_sizes=LAYER_SIZES,
    rounds=ATTACK_ROUNDS,
    m=FAKE_USERS_PER_ROUND,
    n_filler=N_FILLER,
    n_optim_steps=N_OPTIM_STEPS,
    retrain_epochs=RETRAIN_EPOCHS,
    lr=LR,
    device=str(DEVICE),
    eval_fn=eval_fn,
)

# Save
torch.save(attack_model.state_dict(), "checkpoints/amazon/attacked_model.pt")
poisoned_train.to_csv("results/amazon/poisoned_train.csv", index=False)
with open("results/amazon/attack_results.json", "w") as f:
    json.dump({
        "target_item": target_item,
        "n_users": n_users,
        "n_items": n_items,
        "category": "Video_Games",
        "rounds": attack_results,
    }, f, indent=2)

print("\nAttack complete. Checkpoints saved.")

In [ ]:
# Plot attack progress
rounds_keys = sorted(
    [k for k in attack_results if k.startswith("round_")],
    key=lambda x: int(x.split("_")[1]) if x.split("_")[1] != "0" else -1,
)

round_labels = []
hr_vals = []
ndcg_vals = []
for k in rounds_keys:
    m = attack_results[k]
    label = "Clean" if "clean" in k else f"Round {k.split('_')[1]}"
    round_labels.append(label)
    hr_vals.append(m["HR@K"])
    ndcg_vals.append(m["NDCG@K"])

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(round_labels))
ax.plot(x, hr_vals, "g-o", label="HR@10", markersize=7)
ax.plot(x, ndcg_vals, "r-s", label="NDCG@10", markersize=7)
ax.set_xticks(list(x))
ax.set_xticklabels(round_labels)
ax.set_ylabel("Score")
ax.set_title(f"Overall Metrics During Attack — Amazon Video Games (target={target_item})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/amazon/attack_progress.png", dpi=150, bbox_inches="tight")
plt.show()

## Phase 5: Attack Success Evaluation

Key metric: **target item HR@K** — how often the target item appears in top-K recommendations after the attack.

In [ ]:
from src.evaluate import target_item_hit_ratio

def _load_two_tower(ckpt_path):
    """Load a plain TwoTower from a TwoTowerTrainTask checkpoint."""
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    tt_state = {}
    for k, v in state.items():
        if k.startswith("two_tower."):
            tt_state[k[len("two_tower."):]] = v
    if tt_state:
        state = tt_state
    ckpt_n_users = state["ebc.embedding_bags.t_user_id.weight"].shape[0]
    ckpt_n_items = state["ebc.embedding_bags.t_item_id.weight"].shape[0]
    ebc = build_ebc(ckpt_n_users, ckpt_n_items, EMBEDDING_DIM, device=DEVICE)
    tt = TwoTower(ebc, layer_sizes=LAYER_SIZES, device=DEVICE)
    tt.load_state_dict(state, strict=False)
    return tt

clean_model = _load_two_tower("checkpoints/amazon/baseline.pt")
attacked_model = _load_two_tower("checkpoints/amazon/attacked_model.pt")

# Overall metrics
clean_overall = evaluate(clean_model, test_df, train_df, n_items, device=str(DEVICE))
attacked_overall = evaluate(attacked_model, test_df, train_df, n_items, device=str(DEVICE))

# Target item promotion
clean_thr = target_item_hit_ratio(
    clean_model, target_item, test_df, train_df, n_items, device=str(DEVICE)
)
attacked_thr = target_item_hit_ratio(
    attacked_model, target_item, test_df, train_df, n_items, device=str(DEVICE)
)

print("=" * 60)
print("         ATTACK EVALUATION RESULTS (Amazon Video Games)")
print("=" * 60)
print(f"  Target item ID:                {target_item}")
print(f"  Total fake users injected:     {ATTACK_ROUNDS * FAKE_USERS_PER_ROUND}")
print(f"  Poison ratio:                  {ATTACK_ROUNDS * FAKE_USERS_PER_ROUND / n_users * 100:.3f}%")
print(f"  Filler items per fake user:    {N_FILLER}")
print("-" * 60)
print(f"  {'Metric':<30} {'Clean':>12} {'Attacked':>12}")
print("-" * 60)
print(f"  {'Overall HR@10':<30} {clean_overall['HR@K']:>12.4f} {attacked_overall['HR@K']:>12.4f}")
print(f"  {'Overall NDCG@10':<30} {clean_overall['NDCG@K']:>12.4f} {attacked_overall['NDCG@K']:>12.4f}")
print(f"  {'Target Item HR@10':<30} {clean_thr:>12.4f} {attacked_thr:>12.4f}")
print("-" * 60)
if clean_thr > 0:
    print(f"  Target item promotion factor:  {attacked_thr / clean_thr:.1f}x")
else:
    print(f"  Target item promotion:         {clean_thr:.4f} -> {attacked_thr:.4f}")
print("=" * 60)

In [ ]:
# Comparison bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

metrics = ["HR@10", "NDCG@10"]
clean_vals = [clean_overall["HR@K"], clean_overall["NDCG@K"]]
attacked_vals = [attacked_overall["HR@K"], attacked_overall["NDCG@K"]]
x = np.arange(len(metrics))
w = 0.3
ax1.bar(x - w / 2, clean_vals, w, label="Clean", color="steelblue")
ax1.bar(x + w / 2, attacked_vals, w, label="Attacked", color="indianred")
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)
ax1.set_ylabel("Score")
ax1.set_title("Overall Recommendation Quality")
ax1.legend()
ax1.grid(True, alpha=0.3, axis="y")

ax2.bar(["Clean", "Attacked"], [clean_thr, attacked_thr],
        color=["steelblue", "indianred"])
ax2.set_ylabel("Target Item HR@10")
ax2.set_title(f"Target Item Promotion (item {target_item})")
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("results/amazon/attack_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

## Phase 6: Detection (TIA with Threshold Sweep + AUC-ROC)

The TIA method detects fake users by measuring what fraction of each user's interactions are with items similar to the target item in embedding space.

We sweep thresholds at p90/p95/p99 and compute AUC-ROC.

In [ ]:
from src.detect import (
    compute_user_anomaly_scores,
    detect_fake_users,
    sweep_detection_thresholds,
    compute_detection_auc,
)

poisoned = pd.read_csv("results/amazon/poisoned_train.csv")

# Compute anomaly scores
scores = compute_user_anomaly_scores(poisoned, attacked_model, target_item, n_items)

# Separate real vs fake
fake_user_ids = set(poisoned[poisoned["user_id"] >= n_users]["user_id"].unique())
real_scores = scores[~scores.index.isin(fake_user_ids)]
fake_scores = scores[scores.index.isin(fake_user_ids)]

print(f"Real user scores — mean: {real_scores.mean():.4f}, std: {real_scores.std():.4f}")
print(f"Fake user scores — mean: {fake_scores.mean():.4f}, std: {fake_scores.std():.4f}")
print(f"Separation ratio:  {fake_scores.mean() / max(real_scores.mean(), 1e-8):.1f}x")

In [ ]:
# Anomaly score distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(real_scores, bins=50, alpha=0.7,
        label=f"Real users (n={len(real_scores):,})",
        color="steelblue", density=True)
ax.hist(fake_scores, bins=20, alpha=0.7,
        label=f"Fake users (n={len(fake_scores)})",
        color="indianred", density=True)
ax.set_xlabel("Anomaly Score")
ax.set_ylabel("Density")
ax.set_title("TIA Anomaly Score Distribution — Amazon Video Games")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/amazon/detection_scores.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Threshold sweep + AUC-ROC
sweep = sweep_detection_thresholds(
    poisoned, attacked_model, target_item, n_items, n_users,
    percentiles=[90, 95, 99],
)
auc = compute_detection_auc(
    poisoned, attacked_model, target_item, n_items, n_users,
)

print("\n" + "=" * 60)
print("                    DETECTION RESULTS (TIA)")
print("=" * 60)
print(f"  {'Threshold':<12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Flagged':>10}")
print(f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
for label, m in sweep.items():
    print(f"  {label:<12} {m['precision']:>10.4f} {m['recall']:>10.4f} "
          f"{m['f1']:>10.4f} {m['n_flagged']:>10}")
print(f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
print(f"  {'AUC-ROC':<12} {auc:>10.4f}")
print(f"\n  Random baseline precision: {len(fake_user_ids) / len(scores) * 100:.3f}%")
print("=" * 60)

## Phase 7: Fake User Analysis

Inspect what the attack generated — which items did the fake users interact with?

In [ ]:
fake_interactions = poisoned[poisoned["user_id"] >= n_users]
n_fake_users = fake_interactions["user_id"].nunique()
n_fake_interactions = len(fake_interactions)

print(f"Fake users: {n_fake_users}")
print(f"Total fake interactions: {n_fake_interactions:,}")
print(f"Avg interactions per fake user: {n_fake_interactions / n_fake_users:.1f}")
print(f"\nTarget item ({target_item}) appears in "
      f"{(fake_interactions['item_id'] == target_item).sum()}/{n_fake_users} "
      f"fake user profiles")

# Most common items in fake profiles
fake_item_counts = fake_interactions["item_id"].value_counts().head(15)
print(f"\nTop 15 items in fake user profiles:")
for item_id, count in fake_item_counts.items():
    real_pop = item_counts.get(item_id, 0)
    marker = " <-- TARGET" if item_id == target_item else ""
    print(f"  item {item_id:>6}: {count:>3} fake users, "
          f"{real_pop:>5} real interactions{marker}")

In [ ]:
# Compare fake user item popularity distribution vs real users
real_interactions = poisoned[poisoned["user_id"] < n_users]

real_avg_pop = real_interactions.groupby("user_id")["item_id"].apply(
    lambda items: item_counts.reindex(items).mean()
).dropna()
fake_avg_pop = fake_interactions.groupby("user_id")["item_id"].apply(
    lambda items: item_counts.reindex(items).mean()
).dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(real_avg_pop, bins=50, alpha=0.7, label="Real users",
        color="steelblue", density=True)
if len(fake_avg_pop) > 0:
    ax.axvline(fake_avg_pop.mean(), color="indianred", linewidth=2,
               linestyle="--",
               label=f"Fake users (mean={fake_avg_pop.mean():.0f})")
ax.set_xlabel("Avg Item Popularity (# real interactions)")
ax.set_ylabel("Density")
ax.set_title("Item Popularity Profile: Real vs Fake Users")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/amazon/fake_user_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

In [ ]:
# Save all results
summary = {
    "dataset": "Amazon Reviews 2023 — Video_Games (5-core)",
    "n_users": n_users,
    "n_items": n_items,
    "n_interactions": len(df),
    "target_item": target_item,
    "target_item_real_interactions": int(item_counts[target_item]),
    "attack_config": {
        "rounds": ATTACK_ROUNDS,
        "fake_users_per_round": FAKE_USERS_PER_ROUND,
        "total_fake_users": ATTACK_ROUNDS * FAKE_USERS_PER_ROUND,
        "poison_ratio_pct": ATTACK_ROUNDS * FAKE_USERS_PER_ROUND / n_users * 100,
        "n_filler": N_FILLER,
        "n_optim_steps": N_OPTIM_STEPS,
    },
    "baseline": {
        "HR@10": clean_overall["HR@K"],
        "NDCG@10": clean_overall["NDCG@K"],
        "target_HR@10": clean_thr,
    },
    "attacked": {
        "HR@10": attacked_overall["HR@K"],
        "NDCG@10": attacked_overall["NDCG@K"],
        "target_HR@10": attacked_thr,
    },
    "detection": {
        "threshold_sweep": {k: v for k, v in sweep.items()},
        "auc_roc": auc,
        "random_baseline_precision_pct": len(fake_user_ids) / len(scores) * 100,
    },
}

with open("results/amazon/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

In [ ]:
# Download results (Colab)
try:
    from google.colab import files
    import shutil
    shutil.make_archive("amazon_dlattack_results", "zip", "results/amazon")
    files.download("amazon_dlattack_results.zip")
except ImportError:
    print("Not running on Colab — results saved in results/amazon/ directory")